#### Drug-Indication

In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
import os
from pathlib import Path
import pandas as pd

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


# File paths based on your environment
files = {
    'DrugBank': str(BASE / "Output/DB/DrugBank/NAR/DrugBank_Master_Standardized.csv"),
    'DrugCentral': str(BASE / "Output/DB/DrugCentral/NAR/DrugCentral_GPCR_Master_v2.csv"),
    'TTD': str(BASE / "Output/DB/TTD/NAR/TTD_Master_Standardized.csv"),
    'IUPHAR': str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Indications_Standardized.csv"),
    'ChEMBL': str(BASE / "Output/DB/ChEMBL/NAR/ChEMBL_v36_Indications_Standardized.csv")
}

# Helper function to assign a numerical score to a clinical status
def map_status_to_score(status_str):
    if pd.isna(status_str):
        return 0
    s = str(status_str).lower()
    
    if 'approved' in s or s == '4.0' or s == '4':
        return 4
    if 'phase 3' in s or s == '3.0' or s == '3':
        return 3
    if 'phase 2' in s or s == '2.0' or s == '2':
        return 2
    if 'phase 1' in s or s == '1.0' or s == '1':
        return 1
    # Defaults to 0 for 'investigational', 'experimental', 'preclinical', or unknown
    return 0

# Helper function to map the score back to a standardized status label
def get_status_label(score):
    mapping = {4: 'Approved', 3: 'Phase 3', 2: 'Phase 2', 1: 'Phase 1', 0: 'Investigational'}
    return mapping.get(score, 'Investigational')

# Updated function to extract and normalize indication data along with status
def process_indication_data(df, inchi_col, umls_col, source_name, status_col=None, default_score=0):
    # Ensure required columns exist
    if inchi_col not in df.columns or umls_col not in df.columns:
        print(f"Warning: Missing required columns in {source_name}")
        return pd.DataFrame()
    
    # Extract only necessary columns
    cols_to_keep = [inchi_col, umls_col]
    if status_col and status_col in df.columns:
        cols_to_keep.append(status_col)
        
    temp_df = df[cols_to_keep].copy().dropna(subset=[inchi_col, umls_col])
    temp_df.rename(columns={inchi_col: 'InChIKey', umls_col: 'UMLS_CUI'}, inplace=True)
    
    # Evaluate Status Score
    if status_col and status_col in temp_df.columns:
        temp_df['Status_Score'] = temp_df[status_col].apply(map_status_to_score)
    else:
        # For databases like DrugCentral, all listed entries are generally Approved (Score = 4)
        temp_df['Status_Score'] = default_score
        
    # Explode multiple diseases separated by '|' or ','
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].astype(str).str.replace(',', '|')
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].str.split('|')
    temp_df = temp_df.explode('UMLS_CUI')
    
    # Clean up whitespace and drop invalid/empty values
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].str.strip()
    temp_df['InChIKey'] = temp_df['InChIKey'].str.strip()
    temp_df = temp_df[~temp_df['UMLS_CUI'].isin(['', 'UNKNOWN', 'nan', 'NaN'])]
    
    # Record data source
    temp_df['Source'] = source_name
    
    # Keep only relevant columns and drop duplicates
    return temp_df[['InChIKey', 'UMLS_CUI', 'Source', 'Status_Score']].drop_duplicates()


dfs = []

# 1. DrugBank (Status column: 'Groups_ClinicalStatus')
try:
    df_db = pd.read_csv(files['DrugBank'])
    dfs.append(process_indication_data(df_db, 'InChIKey', 'UMLS_CUI', 'DrugBank', status_col='Groups_ClinicalStatus'))
except Exception as e: print("Error loading DrugBank:", e)

# 2. DrugCentral (Implicitly 'Approved', so default_score = 4)
try:
    df_dc = pd.read_csv(files['DrugCentral'])
    dfs.append(process_indication_data(df_dc, 'inchikey', 'Indications_UMLS', 'DrugCentral', default_score=4))
except Exception as e: print("Error loading DrugCentral:", e)

# 3. TTD (Status column: 'Highest_status')
try:
    df_ttd = pd.read_csv(files['TTD'])
    dfs.append(process_indication_data(df_ttd, 'InChIKey', 'Drug_Clinical_Indications_UMLS', 'TTD', status_col='Highest_status'))
except Exception as e: print("Error loading TTD:", e)

# 4. IUPHAR (Clinical Uses listed are generally approved, default_score = 4)
try:
    df_iuphar = pd.read_csv(files['IUPHAR'])
    dfs.append(process_indication_data(df_iuphar, 'InChIKey', 'UMLS_CUI', 'IUPHAR', default_score=4))
except Exception as e: print("Error loading IUPHAR:", e)

# 5. ChEMBL (Status column: 'max_phase_for_ind')
try:
    df_chembl = pd.read_csv(files['ChEMBL'])
    dfs.append(process_indication_data(df_chembl, 'standard_inchi_key', 'umls_cui', 'ChEMBL', status_col='max_phase_for_ind'))
except Exception as e: print("Error loading ChEMBL:", e)

# Concatenate all parsed DataFrames
master_df = pd.concat(dfs, ignore_index=True)

# Group by InChIKey and UMLS_CUI to merge sources and find the highest clinical status
grouped_df = master_df.groupby(['InChIKey', 'UMLS_CUI']).agg(
    Source=('Source', lambda x: '|'.join(sorted(set(x)))),
    Max_Status_Score=('Status_Score', 'max')
).reset_index()

# Convert the maximum numerical score back to the standard text label
grouped_df['Highest_Status'] = grouped_df['Max_Status_Score'].apply(get_status_label)

# Drop the numerical score column as it's no longer needed
grouped_df.drop(columns=['Max_Status_Score'], inplace=True)

# Save the final integrated dataset
output_path = str(BASE / "Output/DB/GPCRactDB/Drug_Indication_Master_Integrated.csv")
grouped_df.to_csv(output_path, index=False)
print(f"Integration complete. File saved to: {output_path}\n")

# ==========================================
# Statistics Output Generation
# ==========================================
print("=== Integrated Drug-Indication Statistics ===")
print(f"Total Unique Drug-Indication Pairs : {len(grouped_df)}")
print(f"Total Unique Drugs (InChIKey)      : {grouped_df['InChIKey'].nunique()}")
print(f"Total Unique Indications (UMLS CUI): {grouped_df['UMLS_CUI'].nunique()}\n")

print("=== Breakdown by Highest Clinical Status ===")
status_counts = grouped_df['Highest_Status'].value_counts()
for status, count in status_counts.items():
    print(f"{status:<15}: {count} pairs")

print("\n=== Breakdown by Database Source Combinations (Top 5) ===")
source_counts = grouped_df['Source'].value_counts().head(5)
for source, count in source_counts.items():
    print(f"{source:<40}: {count} pairs")

#### Target-Indication

In [ ]:
import pandas as pd

# File paths based on your environment
files = {
    'DrugCentral': str(BASE / "Output/DB/DrugCentral/NAR/DrugCentral_GPCR_Master_v2.csv"),
    'TTD': str(BASE / "Output/DB/TTD/NAR/TTD_Master_Standardized.csv"),
    'IUPHAR': str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Indications_Standardized.csv")
}

# Function to extract and normalize target-indication data
def process_target_indication_data(df, target_col, umls_col, source_name):
    # Ensure required columns exist
    if target_col not in df.columns or umls_col not in df.columns:
        print(f"Warning: Missing required columns in {source_name}")
        return pd.DataFrame()
    
    # Extract only necessary columns and rename them to a standard format
    temp_df = df[[target_col, umls_col]].copy().dropna(subset=[target_col, umls_col])
    temp_df.rename(columns={target_col: 'UniProt_AC', umls_col: 'UMLS_CUI'}, inplace=True)
    
    # Explode multiple diseases separated by '|' or ','
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].astype(str).str.replace(',', '|')
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].str.split('|')
    temp_df = temp_df.explode('UMLS_CUI')
    
    # Clean up whitespace and drop invalid/empty values
    temp_df['UMLS_CUI'] = temp_df['UMLS_CUI'].str.strip()
    temp_df['UniProt_AC'] = temp_df['UniProt_AC'].str.strip()
    temp_df = temp_df[~temp_df['UMLS_CUI'].isin(['', 'UNKNOWN', 'nan', 'NaN'])]
    
    # Record data source
    temp_df['Source'] = source_name
    
    # Keep only relevant columns and drop duplicates
    return temp_df[['UniProt_AC', 'UMLS_CUI', 'Source']].drop_duplicates()

dfs = []

# 1. TTD (Target column: 'UNIPROT_AC', Indication column: 'Target_Biological_Diseases_UMLS')
# Note: TTD explicitly defines biological diseases associated with targets
try:
    df_ttd = pd.read_csv(files['TTD'])
    dfs.append(process_target_indication_data(df_ttd, 'UNIPROT_AC', 'Target_Biological_Diseases_UMLS', 'TTD'))
except Exception as e: print("Error loading TTD:", e)

# 2. DrugCentral (Target column: 'ACCESSION', Indication column: 'Indications_UMLS')
try:
    df_dc = pd.read_csv(files['DrugCentral'])
    dfs.append(process_target_indication_data(df_dc, 'ACCESSION', 'Indications_UMLS', 'DrugCentral'))
except Exception as e: print("Error loading DrugCentral:", e)

# 3. IUPHAR (Target column: 'Target UniProt ID', Indication column: 'UMLS_CUI')
try:
    df_iuphar = pd.read_csv(files['IUPHAR'])
    dfs.append(process_target_indication_data(df_iuphar, 'Target UniProt ID', 'UMLS_CUI', 'IUPHAR'))
except Exception as e: print("Error loading IUPHAR:", e)

# Concatenate all parsed DataFrames
master_target_df = pd.concat(dfs, ignore_index=True)

# Group by UniProt_AC and UMLS_CUI to merge sources
grouped_target_df = master_target_df.groupby(['UniProt_AC', 'UMLS_CUI']).agg(
    Source=('Source', lambda x: '|'.join(sorted(set(x))))
).reset_index()

# Save the final integrated dataset
output_path = str(BASE / "Output/DB/GPCRactDB/Target_Indication_Master_Integrated.csv")
grouped_target_df.to_csv(output_path, index=False)
print(f"Integration complete. File saved to: {output_path}\n")

# ==========================================
# Statistics Output Generation
# ==========================================
print("=== Integrated Target-Indication Statistics ===")
print(f"Total Unique Target-Indication Pairs : {len(grouped_target_df)}")
print(f"Total Unique Targets (UniProt AC)    : {grouped_target_df['UniProt_AC'].nunique()}")
print(f"Total Unique Indications (UMLS CUI)  : {grouped_target_df['UMLS_CUI'].nunique()}\n")

print("=== Breakdown by Database Source Combinations ===")
source_counts = grouped_target_df['Source'].value_counts()
for source, count in source_counts.items():
    print(f"{source:<40}: {count} pairs")

In [ ]:
tar = pd.read_csv(str(BASE / "Output/DB/GPCRactDB/Target_Indication_Master_Integrated.csv"))

In [ ]:
tar